# RAG from Scratch to Production — Episode 8/20
## Hybrid search: combining both is harder than it looks

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JakkulaMahendhar/rag-from-scratch/blob/main/notebooks/08_hybrid_search.ipynb)

Episode 7 ended on a promise: embeddings and BM25 miss **different** questions, so together they cover all 11. Combine them, and you get the best of both. That's **hybrid search**, and the [production app](https://github.com/JakkulaMahendhar/rag-chatbot) uses it.

In this episode we test that promise, using **production's exact formula**.

**In this episode:**
1. See why you can't just add the two scores.
2. Rebuild production's hybrid formula, and score it.
3. Find out why "60% vector, 40% BM25" isn't what actually happens.
4. Fix it, and compare with the other popular method (RRF).
5. Ask questions the handbook **can't** answer, and check if any score notices.

**No API key needed.**

## 0. Setup

The same 8 chunks and 11 questions as Episode 7, with Episode 7's fixed tokenizer.

In [ ]:
%pip install -q rank-bm25 sentence-transformers

In [1]:
import re
import numpy as np
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

CHUNKS = [
    "Remote Work\nEmployees may work remotely up to 2 days per week. Wednesday is a mandatory in-office day for all teams. Remote days must be logged in the HR portal by Friday of the previous week.",
    "Annual Leave\nFull-time employees receive 24 days of paid annual leave per calendar year. A maximum of 5 unused days may be carried over into the next year. Book leave with form HR-12.",
    "Parental Leave\nNew parents receive 16 weeks of fully paid leave. Submit form HR-17 at least 8 weeks before the expected start date.",
    "Expenses\nThe meal allowance during business travel is capped at 40 USD per day. Travel expenses are claimed with form EXP-22 within 30 days. Claims above 500 USD need the finance director.",
    "Learning Budget\nEvery employee has an annual learning budget of 1,200 USD for courses and conferences. It does not roll over. Requests are approved by your direct manager.",
    "Equipment\nEvery employee receives a ThinkPad X1 Carbon laptop, refreshed every 3 years. Lost or stolen devices must be reported to IT within 24 hours.",
    "IT Support\nIf the VPN shows error E-4031, your access token has expired: reset it in the Okta portal. For error E-4032, contact the IT helpdesk.",
    "Security\nMulti-factor authentication is mandatory on all company accounts. Passwords must be at least 14 characters long and must never be reused.",
]

QUESTIONS = [
    ("What is form EXP-22 for?",                               "Expenses"),
    ("Which form do I need for parental leave? Is it HR-17?",  "Parental Leave"),
    ("VPN says E-4031, what do I do?",                         "IT Support"),
    ("What does error E-4032 mean?",                           "IT Support"),
    ("Which form is HR-12?",                                   "Annual Leave"),
    ("What laptop model do we get, is it a ThinkPad?",         "Equipment"),
    ("How do I reset my Okta token?",                          "IT Support"),
    ("Can I save unused training money for next year?",        "Learning Budget"),
    ("Do I need 2FA on my work accounts?",                     "Security"),
    ("What do I get for food when travelling for work?",       "Expenses"),
    ("How many remote days per week?",                         "Remote Work"),
]

def title(chunk: str) -> str:
    return chunk.split("\n")[0]

def tokenize(text: str) -> list[str]:
    return re.findall(r"[a-z0-9]+(?:-[a-z0-9]+)*", text.lower())

model = SentenceTransformer("all-MiniLM-L6-v2")
chunk_vectors = model.encode(CHUNKS, normalize_embeddings=True)
bm25 = BM25Okapi([tokenize(c) for c in CHUNKS])


def raw_scores(question: str):
    """The two raw signals for every chunk: cosine similarity and BM25."""
    cosine = chunk_vectors @ model.encode(question, normalize_embeddings=True)
    keyword = np.array(bm25.get_scores(tokenize(question)))
    return cosine, keyword


def accuracy(rank_fn) -> int:
    """How many of the 11 questions get the right chunk in first place."""
    return sum(title(CHUNKS[int(np.argmax(rank_fn(q)))]) == expected for q, expected in QUESTIONS)


print(f"Embeddings alone: {accuracy(lambda q: raw_scores(q)[0])}/11")
print(f"BM25 alone:       {accuracy(lambda q: raw_scores(q)[1])}/11")

Embeddings alone: 10/11
BM25 alone:       9/11


Same as Episode 7: embeddings 10, BM25 9. And between them, every question is found by at least one.

## 1. Why you can't just add the scores

In [2]:
cosine, keyword = raw_scores("Which form is HR-12?")
print(f"{'chunk':<16} {'cosine':>7} {'BM25':>7}")
for c, s, k in zip(CHUNKS, cosine, keyword):
    print(f"{title(c):<16} {s:7.2f} {k:7.2f}")

chunk             cosine    BM25
Remote Work         0.16    0.40
Annual Leave        0.23    1.92
Parental Leave      0.17    0.49
Expenses            0.04    0.85
Learning Budget     0.18    0.00
Equipment           0.04    0.00
IT Support          0.15    0.00
Security            0.15    0.50


Cosine similarity lives between about 0 and 1. BM25 has no fixed maximum: here it reaches about 2, on other questions 5 or more. Add them directly and BM25 wins every time just because its numbers are bigger.

So every hybrid method has to put the two on a **comparable scale** first. How you do that turns out to matter more than anything else.

## 2. Production's formula

The production app (`app/services/hybrid_search.py`):
1. **Vector score** = `1 / (1 + distance)`, where ChromaDB's distance is `2 − 2 × cosine` (Episode 6).
2. **BM25 score** = each chunk's BM25 divided by the **best** BM25 score for this question.
3. **Hybrid** = `0.6 × vector + 0.4 × BM25`.
4. Drop anything below **0.30**.

In [3]:
def production_hybrid(question: str, vector_weight: float = 0.6):
    cosine, keyword = raw_scores(question)
    vector = 1 / (1 + (2 - 2 * cosine))
    keyword = keyword / keyword.max() if keyword.max() > 0 else keyword
    return vector_weight * vector + (1 - vector_weight) * keyword


print(f"Embeddings alone:      {accuracy(lambda q: raw_scores(q)[0])}/11")
print(f"BM25 alone:            {accuracy(lambda q: raw_scores(q)[1])}/11")
print(f"Production hybrid:     {accuracy(production_hybrid)}/11")

Embeddings alone:      10/11
BM25 alone:            9/11
Production hybrid:     9/11


**The hybrid scores worse than embeddings alone.** Combining them didn't give the best of both. Which questions changed?

In [4]:
def top(rank_fn, q):
    return title(CHUNKS[int(np.argmax(rank_fn(q)))])

for q, expected in QUESTIONS:
    emb, hyb = top(lambda x: raw_scores(x)[0], q), top(production_hybrid, q)
    if emb != hyb:
        print(f"{q}\n   embeddings: {emb} {'✅' if emb == expected else '❌'}   hybrid: {hyb} {'✅' if hyb == expected else '❌'}\n")

What is form EXP-22 for?
   embeddings: Security ❌   hybrid: Expenses ✅



Can I save unused training money for next year?
   embeddings: Learning Budget ✅   hybrid: Annual Leave ❌

What do I get for food when travelling for work?
   embeddings: Expenses ✅   hybrid: Remote Work ❌



It fixed EXP-22 (BM25's strength) but **lost** "training money" and "food when travelling" (embeddings' strengths). Net: one worse.

## 3. Why "60% vector" isn't 60%

Look at how much each score **varies** between chunks, for the same question:

In [5]:
vector_spread, bm25_spread = [], []
for q, _ in QUESTIONS:
    cosine, keyword = raw_scores(q)
    vector = 1 / (1 + (2 - 2 * cosine))
    keyword = keyword / keyword.max()
    vector_spread.append(vector.max() - vector.min())
    bm25_spread.append(keyword.max() - keyword.min())

print(f"Vector score: best minus worst chunk, on average  {np.mean(vector_spread):.2f}")
print(f"BM25 score:   best minus worst chunk, on average  {np.mean(bm25_spread):.2f}")

Vector score: best minus worst chunk, on average  0.16
BM25 score:   best minus worst chunk, on average  0.99


The ranking only depends on the **differences** between chunks. And `1 / (1 + distance)` squeezes the vector score into a narrow band (Episode 6: even unrelated text scores 0.33), while BM25-divided-by-max always runs the full 0 to 1.

So the vector side moves the hybrid score by about 0.6 × 0.16 ≈ **0.1**, and BM25 by about 0.4 × 0.99 ≈ **0.4**. **BM25 decides, whatever the weights say.** Here's "training money" up close:

In [6]:
q = "Can I save unused training money for next year?"
cosine, keyword = raw_scores(q)
vector = 1 / (1 + (2 - 2 * cosine))
keyword = keyword / keyword.max()
print(f"{'chunk':<16} {'vector':>7} {'BM25':>6} {'hybrid':>7}")
for c, v, k in sorted(zip(CHUNKS, vector, keyword), key=lambda x: -(0.6 * x[1] + 0.4 * x[2]))[:3]:
    print(f"{title(c):<16} {v:7.2f} {k:6.2f} {0.6 * v + 0.4 * k:7.2f}")

chunk             vector   BM25  hybrid
Annual Leave        0.42   1.00    0.65
Learning Budget     0.46   0.09    0.31
IT Support          0.34   0.09    0.24


Embeddings prefer *Learning Budget*, but only by a few hundredths. BM25 prefers *Annual Leave* ("unused", "next year") by almost a full point. BM25 wins.

And changing the weight barely helps:

In [7]:
for w in [0.2, 0.4, 0.6, 0.8]:
    print(f"vector weight {w}:  {accuracy(lambda q: production_hybrid(q, w))}/11")

vector weight 0.2:  9/11
vector weight 0.4:  9/11


vector weight 0.6:  9/11


vector weight 0.8:  9/11


## 4. Fix 1: put both on the same scale

**Min-max normalization:** for each question, rescale **both** scores so the best chunk gets 1 and the worst gets 0. Now both sides really span 0 to 1, and the weights mean what they say.

In [8]:
def minmax(x):
    return (x - x.min()) / (x.max() - x.min()) if x.max() > x.min() else np.zeros_like(x)

def minmax_hybrid(question: str, vector_weight: float = 0.6):
    cosine, keyword = raw_scores(question)
    return vector_weight * minmax(cosine) + (1 - vector_weight) * minmax(keyword)


for w in [0.4, 0.6, 0.7, 0.8]:
    print(f"min-max, vector weight {w}:  {accuracy(lambda q: minmax_hybrid(q, w))}/11")

min-max, vector weight 0.4:  9/11


min-max, vector weight 0.6:  9/11


min-max, vector weight 0.7:  10/11
min-max, vector weight 0.8:  11/11


**11/11** at a vector weight of 0.8: the hybrid finally gets the best of both.

A warning: I picked 0.8 by testing on the same 11 questions I'm scoring. On your documents the best weight may differ. Tune it on a separate set of test questions, or it'll look better than it is.

## 5. Fix 2: Reciprocal Rank Fusion (RRF)

The other popular method ignores scores completely and uses **rank positions**: each chunk gets `1 / (60 + rank)` from each search, added up. No scaling needed. Many search engines and vector databases offer it built in.

In [9]:
def rrf(question: str, k: int = 60):
    cosine, keyword = raw_scores(question)
    scores = np.zeros(len(CHUNKS))
    for ranking in (np.argsort(-cosine), np.argsort(-keyword)):
        for rank, i in enumerate(ranking, start=1):
            scores[i] += 1 / (k + rank)
    return scores


print(f"RRF: {accuracy(rrf)}/11")

RRF: 9/11


RRF is simple and robust, but here it ties with BM25 alone. It throws away **how much** better one chunk is than another, so on the "training money" question it can't tell that embeddings were fairly confident and BM25 was only matching common words.

| Method | Score (of 11) |
|---|---|
| Embeddings alone | 10 |
| BM25 alone | 9 |
| Production hybrid (0.6 / 0.4) | 9 |
| RRF | 9 |
| **Min-max hybrid (0.8 / 0.2)** | **11** |

## 6. The threshold that never says no

Production drops chunks with a hybrid score below **0.30**, so that a question the documents can't answer gets no chunks, and the bot says "I don't know". Let's ask 4 questions the handbook **can't** answer:

In [10]:
OFF_TOPIC = [
    "What is the capital of France?",
    "Who won the football World Cup in 2022?",
    "How do I bake sourdough bread?",
    "What is the dress code for the office party?",
]

def report(q):
    cosine, keyword = raw_scores(q)
    hybrid = production_hybrid(q)
    passing = int((hybrid >= 0.30).sum())
    print(f"{q[:46]:<48} best hybrid {hybrid.max():.2f}  passing 0.30: {passing}/8   best cosine {cosine.max():.2f}")

print("Can't be answered:")
for q in OFF_TOPIC:
    report(q)
print("\nCan be answered:")
for q, _ in QUESTIONS[:4]:
    report(q)

Can't be answered:
What is the capital of France?                   best hybrid 0.61  passing 0.30: 6/8   best cosine 0.08
Who won the football World Cup in 2022?          best hybrid 0.59  passing 0.30: 3/8   best cosine 0.07
How do I bake sourdough bread?                   best hybrid 0.22  passing 0.30: 0/8   best cosine 0.11
What is the dress code for the office party?     best hybrid 0.62  passing 0.30: 7/8   best cosine 0.16

Can be answered:
What is form EXP-22 for?                         best hybrid 0.62  passing 0.30: 3/8   best cosine 0.15
Which form do I need for parental leave? Is it   best hybrid 0.74  passing 0.30: 4/8   best cosine 0.62


VPN says E-4031, what do I do?                   best hybrid 0.73  passing 0.30: 1/8   best cosine 0.59
What does error E-4032 mean?                     best hybrid 0.66  passing 0.30: 2/8   best cosine 0.35


3 of the 4 off-topic questions score around **0.6**, as high as real questions, and pass 3 to 7 chunks through the 0.30 cutoff. Only *sourdough bread*, which shares no words at all with the handbook, is filtered out.

Why: dividing BM25 by its best score means the top BM25 chunk **always** gets 1.0, even when it only matched "the", "is" and "what". Add a vector score of about 0.33 for unrelated text, and the best chunk scores around 0.4 + 0.6 × 0.33 ≈ **0.6**, for any question that shares even one word with the documents.

Can the **raw** cosine tell them apart instead? Off-topic questions top out around 0.16. But "What is form EXP-22 for?", a real question, has a best cosine of only 0.15. **No single cutoff separates them.**

Retrieval scores are built to **rank** chunks, not to judge whether any chunk is good enough. That job belongs to the **reranker** (Episode 10), which reads question and chunk together, and to the LLM's own "I don't know" rule (Episodes 1 and 15).

## 7. What this means for production

The [production app](https://github.com/JakkulaMahendhar/rag-chatbot):
- **Scales the two scores differently**, so BM25 decides the ranking whatever the 0.6 / 0.4 weights say. Min-max normalizing both (or RRF) makes the weights mean something.
- **Uses a 0.30 threshold that almost never filters**, because the top BM25 chunk always scores 1.0. Off-topic questions still send chunks downstream.
- Also uses the `.split()` tokenizer from Episode 7. With it, the production hybrid scores 8/11 instead of 9.

## Recap

- Combining two retrievers **doesn't automatically** give you the best of both. Here it scored worse than embeddings alone.
- **Scale before you weigh.** If one score barely moves, its weight is meaningless.
- **Measure the combination**, not just the parts. Tune weights on separate test questions.
- **Retrieval scores rank, they don't judge.** Don't rely on them to detect "no answer".

**Next, Episode 9: query enhancement.** Rewriting and expanding the question before searching, and (as promised in the comments on Episode 2) cutting it from 2 LLM calls down to 1.

⭐ Notebooks: https://github.com/JakkulaMahendhar/rag-from-scratch
🏗️ Full production app: https://github.com/JakkulaMahendhar/rag-chatbot
📸 Instagram: [@codingwithmahi](https://instagram.com/codingwithmahi)